In [36]:
%matplotlib qt

import numpy
import mne 
import matplotlib.pyplot as plt
import mne_bids

mne.set_log_level("WARNING")

In [37]:
raw_fname = "/Users/vyompaknikar/Desktop/EEG Dataset/ANALYSE_248/ds000248/sub-01/meg/sub-01_task-audiovisual_run-01_meg.fif"

raw = mne.io.read_raw_fif(raw_fname, preload=True)

In [38]:
raw.info

<Info | 22 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 2 items (MEG 2443, EEG 053)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: Anonymized using a time shift to preserve age at acquisition
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: mne_anonymize
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 line_freq: 60.0
 lowpass: 172.2 Hz
 meas_date: 1921-08-16 19:01:10 UTC
 meas_id: 4 items (dict)
 nchan: 376
 proj_id: 0
 proj_name: mne_anonymize
 projs: PCA-v1: off, PCA-v2: off, PCA-v3: off
 sfreq: 600.6 Hz
>

In [50]:
raw.plot();

In [51]:
raw.annotations

<Annotations | 321 segments: Auditory/Left (72), Auditory/Right (73), BAD_ ...>

In [52]:
raw.plot_psd(fmax=40);

In [43]:
from mne_bids import BIDSPath, read_raw_bids

bids_root = "/Users/vyompaknikar/Desktop/EEG Dataset/ANALYSE_248/ds000248"
bids_path = BIDSPath(subject="01", task="audiovisual", run="01", root=bids_root)

#Read the Raw Data
raw = read_raw_bids(bids_path)
raw.load_data()

<Raw | sub-01_task-audiovisual_run-01_meg.fif, 376 x 166800 (277.7 s), ~481.7 MiB, data loaded>

In [46]:
#Epochs

events = mne.find_events(raw)
event_id = {
    "Auditory/Left": 1,
    "Auditory/Right": 2,
    "Visual/Left": 3,
    "Visual/Right": 4,
}

reject = dict(mag=4e-12, eog=150e-6)

epochs = mne.Epochs(raw, event_id=event_id, events=events, tmin=-0.2, tmax=0.5, baseline=(None, 0), reject=reject)
epochs.load_data()
epochs.plot();
# mne.Epochs?

In [14]:
epochs.plot_drop_log();

In [45]:
evoked = epochs["Auditory/Left"].average()
evoked.plot();

In [16]:
evoked.plot_joint()

[<Figure size 1600x840 with 6 Axes>,
 <Figure size 1600x840 with 6 Axes>,
 <Figure size 1600x840 with 6 Axes>]

In [17]:
epochs_decoding = epochs["Auditory"].copy().pick('mag')
data = epochs_decoding.get_data()
label = epochs_decoding.events[:,2]